In [1]:
import warnings
warnings.filterwarnings("ignore")

from datetime import datetime
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql import Window
from IPython.display import display

pd.DataFrame.iteritems = pd.DataFrame.items

In [2]:
spark = SparkSession.builder\
       .master("local[*]")\
       .appName("VariableSelectionML")\
       .config("spark.executor.memory", "8g")\
       .config("spark.driver.memory", "4g")\
       .getOrCreate()

In [3]:
spark.sparkContext

<SparkContext master=local[*] appName=VariableSelectionML>

In [4]:
# pip install xgboost

In [5]:
df = spark.read.option("header", "true")\
               .option("inferSchema", "true")\
               .option("delimiter", ",")\
               .csv("loan.csv")

window = Window.orderBy(f.lit('A'))
df = df.select(f.row_number().over(window).alias("id"), "*")

display(df.limit(10).toPandas())

,id,loan_amnt,term,int_rate,emp_length,home_ownership,annual_inc,purpose,addr_state,dti,delinq_2yrs,revol_util,total_acc,bad_loan,longest_credit_length,verification_status
0,1,5000,36 months,10.65,10,RENT,24000.0,credit_card,AZ,27.65,0,83.7,9,0,26,verified
1,2,2500,60 months,15.27,0,RENT,30000.0,car,GA,1.00,0,9.4,4,1,12,verified
2,3,2400,36 months,15.96,10,RENT,12252.0,small_business,IL,8.72,0,98.5,10,0,10,not verified
3,4,10000,36 months,13.49,10,RENT,49200.0,other,CA,20.00,0,21.0,37,0,15,verified
4,5,5000,36 months,7.90,3,RENT,36000.0,wedding,AZ,11.20,0,28.3,12,0,7,verified
5,6,3000,36 months,18.64,9,RENT,48000.0,car,CA,5.35,0,87.5,4,0,4,verified
6,7,5600,60 months,21.28,4,OWN,40000.0,small_business,CA,5.55,0,32.6,13,1,7,verified
7,8,5375,60 months,12.69,0,RENT,15000.0,other,TX,18.08,0,36.5,3,1,7,verified
8,9,6500,60 months,14.65,5,OWN,72000.0,debt_consolidation,AZ,16.12,0,20.6,23,0,13,not verified
9,10,12000,36 months,12.69,10,OWN,75000.0,debt_consolidation,CA,10.78,0,67.1,34,0,22,verified


In [6]:
target_name_in_dataset = "bad_loan"

seed = 47  # set seed to your own number for reproducibility

In [7]:
col_list = [colu.lower() for colu in df.columns]

print(f"Columns: {len(col_list)}")

df = df.select(col_list)

print(f"Rows: {df.count()}")

agg_tab = df.agg(f.count(f.lit(1)).alias("count")
        , f.mean(target_name_in_dataset).alias(f"mean_{target_name_in_dataset}"))\
        .withColumn(f"mean_{target_name_in_dataset}", f.round(f"mean_{target_name_in_dataset}", 6))

display(agg_tab.toPandas())

Columns: 16
Rows: 999


,count,mean_bad_loan
0,999,0.194194


In [8]:
# coonstant or id columns to drop

cols_to_ignore =  ["id", "addr_state", target_name_in_dataset]

init_features_list = [colu for colu in col_list if colu.lower() not in cols_to_ignore]

print(f"Features for analysis: {len(init_features_list)}")

Features for analysis: 13


In [9]:
import pyspark.pandas as ps

approx_counts = df.agg(*(f.approx_count_distinct(f.col(c)).alias(c) for c in init_features_list))
psdf = approx_counts.pandas_api()
transposed_psdf = psdf.transpose()
transposed_psdf = transposed_psdf.reset_index()
transposed_psdf.columns = ['col_name', 'count_distinct']
approx_counts_transposed = transposed_psdf.to_spark()

counts = df.agg(*(f.count(f.col(c)).alias(c) for c in init_features_list))
psdf = counts.pandas_api()
transposed_psdf = psdf.transpose()
transposed_psdf = transposed_psdf.reset_index()
transposed_psdf.columns = ['col_name', 'count']
counts_transposed = transposed_psdf.to_spark()

approx_counts_transposed = approx_counts_transposed.join(counts_transposed, on='col_name')

n = df.count()
approx_counts_transposed = approx_counts_transposed.withColumn("total", f.lit(n))
approx_counts_transposed = approx_counts_transposed.withColumn("missing", f.col("total") - f.col("count"))
approx_counts_transposed = approx_counts_transposed.withColumn("pct_missing", f.col("missing")/f.col("total"))
approx_counts_transposed = approx_counts_transposed.withColumn("pct_missing", f.round("pct_missing", 4))
approx_counts_transposed = approx_counts_transposed.withColumn("pct_not_missing", f.col("count")/f.col("total"))
approx_counts_transposed = approx_counts_transposed.withColumn("pct_not_missing", f.round("pct_not_missing", 4))
approx_counts_transposed = approx_counts_transposed.select("col_name", "count_distinct", "missing", "pct_missing", "pct_not_missing", "total")

approx_counts_transposed.show(20, False)

+---------------------+--------------+-------+-----------+---------------+-----+
|col_name             |count_distinct|missing|pct_missing|pct_not_missing|total|
+---------------------+--------------+-------+-----------+---------------+-----+
|annual_inc           |281           |0      |0.0        |1.0            |999  |
|delinq_2yrs          |4             |0      |0.0        |1.0            |999  |
|dti                  |847           |0      |0.0        |1.0            |999  |
|emp_length           |11            |17     |0.017      |0.983          |999  |
|home_ownership       |3             |0      |0.0        |1.0            |999  |
|int_rate             |32            |0      |0.0        |1.0            |999  |
|loan_amnt            |220           |0      |0.0        |1.0            |999  |
|longest_credit_length|35            |0      |0.0        |1.0            |999  |
|purpose              |13            |0      |0.0        |1.0            |999  |
|revol_util           |605  

In [10]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as f

def woe_categorical(
    df: DataFrame,
    feature_col: str,
    target_col: str,
    event_value: int = 1,
    eps: float = 0.5
) -> DataFrame:
    """
    WoE for a categorical feature column in a Spark DataFrame.

    Returns a DataFrame with:
    - category, total, events, non_events
    - dist_event, dist_non_event
    - woe, iv_component
    """
    if event_value is None:
        event_value = (
            df.agg(f.max(f.col(target_col)).alias("event_value"))
            .first()["event_value"]
        )

    # Pre-process target and feature casting
    base_df = df.select(
        f.col(feature_col).alias("category"),
        f.when(f.col(target_col) == f.lit(event_value), 1)
         .otherwise(0)
         .alias("is_event")
    )

    # Aggregate counts per category
    counts_df = (
        base_df.groupBy("category")
        .agg(
            f.count(f.lit(1)).alias("total"),
            f.sum("is_event").alias("events")
        )
        .withColumn("non_events", f.col("total") - f.col("events"))
    )

    # Calculate global totals for distributions
    totals = counts_df.select(
        f.sum("events").alias("total_events"),
        f.sum("non_events").alias("total_non_events")
    ).first()

    total_ev = totals["total_events"]
    total_nev = totals["total_non_events"]

    # Compute WoE and IV components
    result = (
        counts_df
        .withColumn("dist_event", (f.col("events") + eps) / (total_ev + eps))
        .withColumn("dist_non_event", (f.col("non_events") + eps) / (total_nev + eps))
        .withColumn("woe", f.log(f.col("dist_non_event") / f.col("dist_event")))
        .withColumn(
            "iv_component", 
            (f.col("dist_non_event") - f.col("dist_event")) * f.col("woe")
        )
        .select(
            f.col("category").alias(feature_col),
            "total", "events", "non_events",
            f.round("dist_event", 6).alias("dist_event"),
            f.round("dist_non_event", 6).alias("dist_non_event"),
            f.round("woe", 6).alias("woe"),
            f.round("iv_component", 6).alias("iv_component")
        )
        .orderBy("woe")
    )

    return result

In [11]:
import logging
logging.getLogger("XGBoost-PySpark").setLevel(logging.WARNING)

from pyspark.sql.types import StringType
from pyspark.ml.feature import Bucketizer
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

approx_counts_transposed = approx_counts_transposed.withColumn("gini", f.lit(None))
approx_counts_transposed = approx_counts_transposed.withColumn("reason", f.lit(None))
app_counts = approx_counts_transposed.toPandas()

string_cols = [x.name for x in df.select(init_features_list).schema.fields if isinstance(x.dataType, StringType)]

print("--- {} Start".format(datetime.now().strftime("%Y/%m/%d %H:%M:%S")))

for i in range(len(app_counts)):
    feature = app_counts.iloc[i, app_counts.columns.get_loc('col_name')]
    n = app_counts.iloc[i, app_counts.columns.get_loc('count_distinct')]
    
    if ((n >= 2) and (feature in string_cols)):
        try:
            missing_label = "__MISSING__"
            df_copy = df.select(feature, target_name_in_dataset)\
                        .fillna({feature: missing_label})
            
            feature_woe = f"{feature}_woe"
            
            df_woe = woe_categorical(df_copy, feature, target_name_in_dataset)\
                .select(feature, f.col("woe").alias(feature_woe))
            
            df_copy = df_copy.join(df_woe, on=feature).drop(feature)
            
            assembler = VectorAssembler(
                inputCols=[feature_woe],
                outputCol="features"
            )
            
            lr = LogisticRegression(
                featuresCol = "features",
                labelCol = target_name_in_dataset,
                family = "binomial",
                regParam = 0.0,
                elasticNetParam = 0.0
            )
            
            pipeline = Pipeline(stages=[assembler, lr])
            model = pipeline.fit(df_copy)
            predictions = model.transform(df_copy)
            
            evaluator = BinaryClassificationEvaluator(labelCol = target_name_in_dataset, metricName="areaUnderROC")
            auc = evaluator.evaluate(predictions)
            gini = 2.0 * auc - 1.0
            
            app_counts.iloc[i, app_counts.columns.get_loc('reason')] = "ok"
            app_counts.iloc[i, app_counts.columns.get_loc('gini')] = round(gini, 6)

        except Exception as e:
            app_counts.iloc[i, app_counts.columns.get_loc('reason')] = f"issue with feature {feature}: {e}"
            app_counts.iloc[i, app_counts.columns.get_loc('gini')] = 0.0

    elif ((n >= 2) and (feature not in string_cols)):
        try:
            df_copy = df.select(feature, target_name_in_dataset)

            xgb_classifier = SparkXGBClassifier(
                            features_col = [feature],
                            label_col = target_name_in_dataset,
                            n_estimators = 1,
                            max_depth = 3,
                            learning_rate = 0.1,
                            num_workers = spark.sparkContext.defaultParallelism,
                            use_gpu = True,
                            verbosity = 0 
                        )
            
            model = xgb_classifier.fit(df_copy)

            booster = model.get_booster()
            df_split = booster.get_split_value_histogram(feature = feature)
            split_list = df_split['SplitValue'].to_list()
            splits = [float('-inf')] + split_list + [float('inf')]
            splits = list(set(splits))
            splits.sort()

            feature_bin = f"{feature}_bin"

            bucketizer = Bucketizer(splits=splits, inputCol = feature, outputCol = feature_bin)
            df_copy_binned = bucketizer.transform(df_copy)
            df_copy_binned = df_copy_binned.withColumn(feature_bin, f.col(feature_bin) + f.lit(1)).fillna({feature_bin: 0}).drop(feature)

            feature_woe = f"{feature}_woe"
            
            df_woe = woe_categorical(df_copy_binned, feature_bin, target_name_in_dataset)\
                .select(feature_bin, f.col("woe").alias(feature_woe))
            
            df_copy_binned = df_copy_binned.join(df_woe, on=feature_bin).drop(feature_bin)
            
            assembler = VectorAssembler(
                inputCols=[feature_woe],
                outputCol="features"
            )
            
            lr = LogisticRegression(
                featuresCol = "features",
                labelCol = target_name_in_dataset,
                family = "binomial",
                regParam = 0.0,
                elasticNetParam = 0.0
            )
            
            pipeline = Pipeline(stages=[assembler, lr])
            model = pipeline.fit(df_copy_binned)
            predictions = model.transform(df_copy_binned)
            
            evaluator = BinaryClassificationEvaluator(labelCol = target_name_in_dataset, metricName="areaUnderROC")
            auc = evaluator.evaluate(predictions)
            gini = 2.0 * auc - 1.0
            
            app_counts.iloc[i, app_counts.columns.get_loc('reason')] = "ok"
            app_counts.iloc[i, app_counts.columns.get_loc('gini')] = round(gini, 6)

        except Exception as e:
            app_counts.iloc[i, app_counts.columns.get_loc('reason')] = f"issue with feature {feature}: {e}"
            app_counts.iloc[i, app_counts.columns.get_loc('gini')] = 0.0
        
    else:
        app_counts.iloc[i, app_counts.columns.get_loc('reason')] = "const value"
        app_counts.iloc[i, app_counts.columns.get_loc('trai_gini')] = 0.0
        app_counts.iloc[i, app_counts.columns.get_loc('cv_gini')] = 0.0

print("--- {} End".format(datetime.now().strftime("%Y/%m/%d %H:%M:%S")))        
 
approx_counts_transposed = spark.createDataFrame(app_counts)
approx_counts_transposed.orderBy(f.col('gini').desc()).show(20, False)

--- 2026/03/28 23:18:21 Start
--- 2026/03/28 23:19:14 End
+---------------------+--------------+-------+-----------+---------------+-----+--------+------+
|col_name             |count_distinct|missing|pct_missing|pct_not_missing|total|gini    |reason|
+---------------------+--------------+-------+-----------+---------------+-----+--------+------+
|int_rate             |32            |0      |0.0        |1.0            |999  |0.300986|ok    |
|term                 |2             |0      |0.0        |1.0            |999  |0.233246|ok    |
|purpose              |13            |0      |0.0        |1.0            |999  |0.180444|ok    |
|total_acc            |55            |0      |0.0        |1.0            |999  |0.13808 |ok    |
|revol_util           |605           |0      |0.0        |1.0            |999  |0.126907|ok    |
|dti                  |847           |0      |0.0        |1.0            |999  |0.112704|ok    |
|loan_amnt            |220           |0      |0.0        |1.0        